In [1]:
import os
import cv2
import timm
import torch
import random
import numpy as np
import pandas as pd

from PIL import Image

from sklearn.metrics import f1_score

from torch import nn
from torch.utils.data import Dataset, DataLoader

import albumentations as A
from albumentations.pytorch import ToTensorV2

import pandas as pd
from sklearn.utils.class_weight import compute_class_weight
from tqdm.auto import tqdm
from sklearn.metrics import f1_score
import time

d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [3]:
CFG = {
    "model_name": "swinv2_tiny_window16_256",
    "img_size": 256,
    "batch_size": 32,
    "epochs": 30,
    "patience": 5,
    "lr": 2e-4,
    "weight_decay": 1e-4,
    "num_classes": 3,
    "num_workers": 0,
    "fold": 0,
    "device": "cuda"
}

In [4]:
train_transform = A.Compose([

    A.LongestMaxSize(max_size=CFG["img_size"]),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.HorizontalFlip(p=0.5),

    A.ShiftScaleRotate(
        shift_limit=0.05,
        scale_limit=0.10,
        rotate_limit=10,
        p=0.5,
    ),

    A.RandomBrightnessContrast(
        brightness_limit=0.15,
        contrast_limit=0.15,
        p=0.3,
    ),

    A.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2,
        hue=0.1,
        p=0.5,
    ),

    A.CLAHE(
        clip_limit=2,
        tile_grid_size=(8,8),
        p=0.5,
    ),

    A.Normalize(),

    ToTensorV2()


])

d:\BDC_2026_PP\P_PP\poetri\.venv\Lib\site-packages\albumentations\core\validation.py:114: UserWarning: ShiftScaleRotate is a special case of Affine transform. Please use Affine transform instead.
  original_init(self, **validated_kwargs)


In [5]:
valid_transform = A.Compose([

    A.LongestMaxSize(max_size=CFG["img_size"]),

    A.PadIfNeeded(
        min_height=CFG["img_size"],
        min_width=CFG["img_size"],
        border_mode=cv2.BORDER_CONSTANT
    ),

    A.Normalize(),

    ToTensorV2()

])

In [6]:
class WasteDataset(Dataset):

    def __init__(self, df, transforms=None):
        self.df = df.reset_index(drop=True)
        self.transforms = transforms

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = cv2.imread(row.filepath)
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

        if self.transforms:
            image = self.transforms(image=image)["image"]

        label = row.label
        return image, label

In [7]:
df = pd.read_csv("train_folds.csv")

print(df.shape)
df.head()

(24853, 4)


,filepath,label,class_name,fold
0,..\data\train\0_Recyclable\R_1.jpg,0,0_Recyclable,2
1,..\data\train\0_Recyclable\R_10.jpg,0,0_Recyclable,2
2,..\data\train\0_Recyclable\R_100.jpg,0,0_Recyclable,2
3,..\data\train\0_Recyclable\R_1000.jpg,0,0_Recyclable,3
4,..\data\train\0_Recyclable\R_1001.jpg,0,0_Recyclable,3


In [8]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device):

    model.train()

    running_loss = 0

    pbar = tqdm(loader, desc="Train")

    for images, labels in pbar:

        images = images.to(device, non_blocking=True)
        labels = labels.to(device)

        optimizer.zero_grad()

        with torch.amp.autocast("cuda"):

            outputs = model(images)

            loss = criterion(outputs, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item()

        pbar.set_postfix(loss=f"{loss.item():.4f}")

    return running_loss / len(loader)

In [9]:
def valid_one_epoch(model, loader, criterion, device):

    model.eval()

    running_loss = 0

    preds = []
    labels_all = []

    pbar = tqdm(loader, desc="Valid")

    with torch.no_grad():

        for images, labels in pbar:

            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)

            loss = criterion(outputs, labels)

            running_loss += loss.item()

            pred = outputs.argmax(1)

            preds.extend(pred.cpu().numpy())
            labels_all.extend(labels.cpu().numpy())

            pbar.set_postfix(loss=f"{loss.item():.4f}")

    macro_f1 = f1_score(
        labels_all,
        preds,
        average="macro"
    )

    return running_loss / len(loader), macro_f1

In [10]:
CFG["fold"] = 0

In [11]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

print(train_df.shape)
print(valid_df.shape)

(19882, 4)
(4971, 4)


In [12]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [13]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [14]:
print(len(train_loader))
print(len(valid_loader))

622
156


In [15]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

print(class_weights)

tensor([0.8752, 2.1079, 0.7230], device='cuda:0')


In [16]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [17]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

In [18]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

In [19]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

In [20]:
scaler = torch.amp.GradScaler("cuda")

In [21]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"swin_best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"swin_last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [01:42<00:00,  1.52it/s, loss=0.5182]


Epoch [1/30] | Train Loss: 0.3537 | Valid Loss: 0.3400 | Macro F1: 0.9540 | Time: 1745.0s
Best model saved! Macro F1 = 0.95399


Valid: 100%|██████████| 156/156 [01:18<00:00,  1.99it/s, loss=0.4022]


Epoch [2/30] | Train Loss: 0.2957 | Valid Loss: 0.3054 | Macro F1: 0.9711 | Time: 1603.8s
Best model saved! Macro F1 = 0.97115


Valid: 100%|██████████| 156/156 [01:16<00:00,  2.05it/s, loss=0.3241]


Epoch [3/30] | Train Loss: 0.2793 | Valid Loss: 0.3133 | Macro F1: 0.9674 | Time: 1604.4s
No improvement (1/5)


... (4 outputs left out)

Valid: 100%|██████████| 156/156 [01:48<00:00,  1.43it/s, loss=0.3070]


Epoch [6/30] | Train Loss: 0.2626 | Valid Loss: 0.3039 | Macro F1: 0.9700 | Time: 1677.9s
No improvement (4/5)


Valid: 100%|██████████| 156/156 [01:17<00:00,  2.00it/s, loss=0.3019]

Epoch [7/30] | Train Loss: 0.2491 | Valid Loss: 0.3185 | Macro F1: 0.9659 | Time: 1691.0s
No improvement (5/5)
Early Stopping
Training Finished!
Best Macro F1 : 0.97115


In [22]:
CFG["fold"] = 1

In [23]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

print(train_df.shape)
print(valid_df.shape)

(19882, 4)
(4971, 4)


In [24]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [25]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [26]:
print(len(train_loader))
print(len(valid_loader))

622
156


In [27]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

print(class_weights)

tensor([0.8751, 2.1079, 0.7231], device='cuda:0')


In [28]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [29]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

In [30]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

In [31]:
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

In [32]:
scaler = torch.amp.GradScaler("cuda")

In [33]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"swin_best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"swin_last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [01:18<00:00,  1.98it/s, loss=0.4619]


Epoch [1/30] | Train Loss: 0.3449 | Valid Loss: 0.3376 | Macro F1: 0.9617 | Time: 1549.4s
Best model saved! Macro F1 = 0.96173


Valid: 100%|██████████| 156/156 [01:21<00:00,  1.91it/s, loss=0.7697]


Epoch [2/30] | Train Loss: 0.2965 | Valid Loss: 0.3396 | Macro F1: 0.9580 | Time: 1544.9s
No improvement (1/5)


Valid: 100%|██████████| 156/156 [01:19<00:00,  1.97it/s, loss=0.5248]


Epoch [3/30] | Train Loss: 0.2857 | Valid Loss: 0.3029 | Macro F1: 0.9724 | Time: 1541.2s
Best model saved! Macro F1 = 0.97242


... (10 outputs left out)

Valid: 100%|██████████| 156/156 [00:37<00:00,  4.12it/s, loss=0.9356]


Epoch [9/30] | Train Loss: 0.2453 | Valid Loss: 0.3118 | Macro F1: 0.9712 | Time: 617.3s
No improvement (4/5)


Valid: 100%|██████████| 156/156 [00:36<00:00,  4.33it/s, loss=0.6518]

Epoch [10/30] | Train Loss: 0.2358 | Valid Loss: 0.3009 | Macro F1: 0.9702 | Time: 613.8s
No improvement (5/5)
Early Stopping
Training Finished!
Best Macro F1 : 0.97381


In [34]:
CFG["fold"] = 2

In [35]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [36]:
train_dataset = WasteDataset(train_df, train_transform)
valid_dataset = WasteDataset(valid_df, valid_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [37]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [38]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [39]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [40]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"swin_best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"swin_last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:42<00:00,  3.64it/s, loss=0.3568]


Epoch [1/30] | Train Loss: 0.3527 | Valid Loss: 0.3325 | Macro F1: 0.9600 | Time: 652.2s
Best model saved! Macro F1 = 0.96003


Valid: 100%|██████████| 156/156 [00:42<00:00,  3.63it/s, loss=0.6679]


Epoch [2/30] | Train Loss: 0.2990 | Valid Loss: 0.3214 | Macro F1: 0.9606 | Time: 651.2s
Best model saved! Macro F1 = 0.96056


Valid: 100%|██████████| 156/156 [00:42<00:00,  3.67it/s, loss=0.6568]


Epoch [3/30] | Train Loss: 0.2818 | Valid Loss: 0.3154 | Macro F1: 0.9679 | Time: 648.7s
Best model saved! Macro F1 = 0.96788


... (20 outputs left out)

Valid: 100%|██████████| 156/156 [00:40<00:00,  3.86it/s, loss=0.9949]


Epoch [14/30] | Train Loss: 0.2248 | Valid Loss: 0.3098 | Macro F1: 0.9704 | Time: 644.8s
No improvement (4/5)


Valid: 100%|██████████| 156/156 [00:40<00:00,  3.86it/s, loss=0.6619]

Epoch [15/30] | Train Loss: 0.2185 | Valid Loss: 0.3023 | Macro F1: 0.9754 | Time: 644.2s
No improvement (5/5)
Early Stopping
Training Finished!
Best Macro F1 : 0.97625


In [41]:
CFG["fold"] = 3

In [42]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [43]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [44]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [45]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [46]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [47]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [48]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"swin_best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"swin_last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:41<00:00,  3.80it/s, loss=0.7704]


Epoch [1/30] | Train Loss: 0.3530 | Valid Loss: 0.3726 | Macro F1: 0.9360 | Time: 574.5s
Best model saved! Macro F1 = 0.93598


Valid: 100%|██████████| 156/156 [00:41<00:00,  3.80it/s, loss=0.4023]


Epoch [2/30] | Train Loss: 0.2941 | Valid Loss: 0.3190 | Macro F1: 0.9637 | Time: 574.1s
Best model saved! Macro F1 = 0.96372


Valid: 100%|██████████| 156/156 [00:41<00:00,  3.80it/s, loss=0.4824]


Epoch [3/30] | Train Loss: 0.2802 | Valid Loss: 0.3289 | Macro F1: 0.9582 | Time: 574.6s
No improvement (1/5)


... (26 outputs left out)

Valid: 100%|██████████| 156/156 [00:40<00:00,  3.81it/s, loss=0.3608]


Epoch [17/30] | Train Loss: 0.2160 | Valid Loss: 0.3024 | Macro F1: 0.9785 | Time: 570.1s
No improvement (4/5)


Valid: 100%|██████████| 156/156 [00:40<00:00,  3.81it/s, loss=0.6956]


Epoch [18/30] | Train Loss: 0.2156 | Valid Loss: 0.3061 | Macro F1: 0.9771 | Time: 570.1s
No improvement (5/5)
Early Stopping
Training Finished!
Best Macro F1 : 0.97873


In [49]:
CFG["fold"] = 4

In [50]:
fold = CFG["fold"]

train_df = df[df["fold"] != fold].reset_index(drop=True)
valid_df = df[df["fold"] == fold].reset_index(drop=True)

In [51]:
train_dataset = WasteDataset(
    train_df,
    train_transform
)

valid_dataset = WasteDataset(
    valid_df,
    valid_transform
)

In [52]:
train_loader = DataLoader(
    train_dataset,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=0,
    pin_memory=True
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=0,
    pin_memory=True
)

In [53]:
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=np.array([0,1,2]),
    y=train_df["label"]
)

class_weights = torch.tensor(
    class_weights,
    dtype=torch.float32
).to(CFG["device"])

In [54]:
model = timm.create_model(
    CFG["model_name"],
    pretrained=True,
    num_classes=CFG["num_classes"]
).to(CFG["device"])

In [55]:
criterion = nn.CrossEntropyLoss(
    weight=class_weights,
    label_smoothing=0.05
)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=CFG["lr"],
    weight_decay=CFG["weight_decay"]
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=CFG["epochs"]
)

scaler = torch.amp.GradScaler("cuda")

In [56]:
best_f1 = 0.0
counter = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss = train_one_epoch(
        model=model,
        loader=train_loader,
        optimizer=optimizer,
        criterion=criterion,
        scaler=scaler,
        device=CFG["device"]
    )

    valid_loss, valid_f1 = valid_one_epoch(
        model=model,
        loader=valid_loader,
        criterion=criterion,
        device=CFG["device"]
    )

    scheduler.step()

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] | "
        f"Train Loss: {train_loss:.4f} | "
        f"Valid Loss: {valid_loss:.4f} | "
        f"Macro F1: {valid_f1:.4f} | "
        f"Time: {elapsed:.1f}s"
    )

    if valid_f1 > best_f1:

        best_f1 = valid_f1
        counter = 0

        torch.save(
            model.state_dict(),
            f"swin_best_fold{CFG['fold']}.pth"
        )

        print(f"Best model saved! Macro F1 = {best_f1:.5f}")

    else:

        counter += 1
        print(f"No improvement ({counter}/{CFG['patience']})")

    if counter >= CFG["patience"]:

        print("Early Stopping")
        break

torch.save(
    model.state_dict(),
    f"swin_last_fold{CFG['fold']}.pth"
)

print("Training Finished!")
print(f"Best Macro F1 : {best_f1:.5f}")

Valid: 100%|██████████| 156/156 [00:40<00:00,  3.90it/s, loss=0.3014]


Epoch [1/30] | Train Loss: 0.3503 | Valid Loss: 0.3320 | Macro F1: 0.9616 | Time: 666.4s
Best model saved! Macro F1 = 0.96156


Valid: 100%|██████████| 156/156 [00:39<00:00,  3.90it/s, loss=0.3161]


Epoch [2/30] | Train Loss: 0.2987 | Valid Loss: 0.3467 | Macro F1: 0.9530 | Time: 667.7s
No improvement (1/5)


Valid: 100%|██████████| 156/156 [00:40<00:00,  3.90it/s, loss=0.3134]


Epoch [3/30] | Train Loss: 0.2753 | Valid Loss: 0.3173 | Macro F1: 0.9682 | Time: 665.7s
Best model saved! Macro F1 = 0.96818


... (12 outputs left out)

Valid: 100%|██████████| 156/156 [00:39<00:00,  3.90it/s, loss=0.3017]


Epoch [10/30] | Train Loss: 0.2380 | Valid Loss: 0.3130 | Macro F1: 0.9735 | Time: 665.6s
No improvement (4/5)


Valid: 100%|██████████| 156/156 [00:39<00:00,  3.91it/s, loss=0.2967]


Epoch [11/30] | Train Loss: 0.2372 | Valid Loss: 0.3163 | Macro F1: 0.9706 | Time: 665.1s
No improvement (5/5)
Early Stopping
Training Finished!
Best Macro F1 : 0.97415
